In [6]:
#Retrievl and Chain with Langchain

##PDFLoader
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('attention.pdf')
docs=loader.load()
docs



[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-03-13T10:31:53+00:00', 'source': 'attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="What is Generative AI\nWhat is Generative AI and How Transformers Make it Possible\nAn Introduction to Generative AI and the Role of Transformers\nBy Bhaskar jha | Feb 29, 2024\nRecent breakthroughs in the field of GEN AI are taking the world by storm and have the potential to\nchange how we approach content creation drastically. Generative AI is a machine?s ability to create\nnew content, including text, images, audio, video, code, and simulations.\nGenerative AI systems fall under the broad category of Artificial Intelligence and Machine learning.\nWhat is GEN AI?\nGen AI is a subset of Deep learning (a subset of Machine learning). It uses artificial neural networks\nand can process labeled and unlabeled data using supervised, unsupervised, and semi-supervised\nmethods.\nMachine learning mod

In [7]:
##Using Splitter
from langchain_text_splitters  import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunk_documents=text_splitter.split_documents(docs)
chunk_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-03-13T10:31:53+00:00', 'source': 'attention.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='What is Generative AI\nWhat is Generative AI and How Transformers Make it Possible\nAn Introduction to Generative AI and the Role of Transformers\nBy Bhaskar jha | Feb 29, 2024\nRecent breakthroughs in the field of GEN AI are taking the world by storm and have the potential to\nchange how we approach content creation drastically. Generative AI is a machine?s ability to create\nnew content, including text, images, audio, video, code, and simulations.\nGenerative AI systems fall under the broad category of Artificial Intelligence and Machine learning.\nWhat is GEN AI?\nGen AI is a subset of Deep learning (a subset of Machine learning). It uses artificial neural networks\nand can process labeled and unlabeled data using supervised, unsupervised, and semi-supervised\nmethods.\nMachine learning mod

In [8]:
##Vector embedding  andvector search
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

#gemini-2.5-flash-lite
os.environ["GEMINI_API_KEY"] =os.getenv("GEMINI_API_KEY")
db=FAISS.from_documents(chunk_documents,GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview"))
db

In [9]:
##Query fomthe attention.pdf file , as those are stored in vector db above...
query=" NLP models like Recurrent Neural Networks (RNN) and Long"
retrieved_result=db.similarity_search(query)
print(retrieved_result[0].page_content)

Before Transformers, traditional NLP models like Recurrent Neural Networks (RNN) and Long
Short-Term Memory (LSTM) processed text sequentially. They would read one word at a time,
keeping track of previous words to understand the context. However, this sequential processing had
several limitations:
1. Difficulty in capturing long-range dependencies: As sentences got longer, it became harder for
RNNs to remember information from the beginning of the sentence.
2. Computational inefficiency: Sequential processing is slow because each word depends on the
previous one, making it difficult to parallelize training.
How does the Transformer work?
Page 1


In [10]:
#ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
    Answer the following question based only on the provided context
    Thik stepby step before providing a detailed answer.
    I will tip you $1000  if the user finds the answer helpful.
    <context>
    {context}
    </context>

    Question:{input}
    """)
    

In [11]:
##chain

groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key
modelGroq=ChatGroq(model="llama-3.1-8b-instant")

In [12]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(modelGroq,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context\n    Thik stepby step before providing a detailed answer.\n    I will tip you $1000  if the user finds the answer helpful.\n    <context>\n    {context}\n    </context>\n\n    Question:{input}\n    '), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.r

In [16]:
####Retriever
"""
Retrievers: A retriever is an interface that returns document given
an unstructured query.It is more general than a vetor store.
A retriever doesnot need to be able to store documents, onlyro
return them.Vector stores ca be used a backbone of a retriever ,but 
there are other types  as well.
https://python.langchain.com/docs/modules/data_connection/retrievers/
"""

retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001DD9F7E7230>, search_kwargs={})

In [15]:
"""
Retrieval chain:A retrieval chain is a structured, multi-step pipeline
in AI frameworks (primarily LangChain) that connects a Large Language Model (LLM)
with external data sources to generate informed responses,
forming the core of Retrieval-Augmented Generation (RAG). 
https://python.langchain.com/docs/modules/chains/
"""

from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [20]:
resposne = retrieval_chain.invoke({"input":" NLP models like Recurrent Neural Networks (RNN)"})
resposne['answer']

'To answer the question, I will break it down step by step:\n\n1. The question mentions "NLP models like Recurrent Neural Networks (RNN)".\n2. I need to recall the context provided, which talks about NLP models, specifically mentioning RNN and Long Short-Term Memory (LSTM) as traditional models.\n3. According to the context, these traditional models (RNN and LSTM) process text sequentially, reading one word at a time.\n\nNow, providing a detailed answer:\n\nNLP models like Recurrent Neural Networks (RNN) and Long Short-Term Memory (LSTM) process text sequentially, reading one word at a time and keeping track of previous words to understand the context. This sequential processing had several limitations, including difficulty in capturing long-range dependencies and computational inefficiency.'